# What Problem Does the Reducer Solve in LangGraph?

## Introduction

Before understanding **Reducers**, it's important to understand the problem they solve.

In a LangGraph workflow, every node can update the **shared state**. This works well when only **one node** updates a state variable. However, problems arise when **multiple nodes update the same state field simultaneously**, especially in **parallel workflows**.

Without a reducer, LangGraph doesn't know how to combine these updates.

---

# The Problem

Imagine a workflow where three nodes run in parallel.

```text
                 Start
                    │
        ┌───────────┼───────────┐
        ▼           ▼           ▼
     Node A      Node B      Node C
```

Each node generates some data.

### Node A

```python
{
    "messages": ["Hello"]
}
```

### Node B

```python
{
    "messages": ["How are you?"]
}
```

### Node C

```python
{
    "messages": ["Welcome!"]
}
```

Now all three nodes are trying to update the same state key:

```python
messages
```

The question is:

> **How should LangGraph combine these three updates?**

---

# What Happens Without a Reducer?

Without a reducer, LangGraph has no merge strategy.

One update may overwrite another.

For example:

```text
Node A finishes first

messages = ["Hello"]
```

Then Node B finishes:

```text
messages = ["How are you?"]
```

Finally Node C finishes:

```text
messages = ["Welcome!"]
```

The final state becomes:

```python
{
    "messages": ["Welcome!"]
}
```

The previous messages are lost.

---

# Visual Representation

```text
            Parallel Execution

      Node A → ["Hello"]
             \
              \
      Node B → ["How are you?"] ----> Shared State
              /
             /
      Node C → ["Welcome!"]
```

Without a reducer:

```text
Shared State

↓

["Welcome!"]
```

The outputs from Node A and Node B are overwritten.

---

# Why Is This a Problem?

Imagine building a chatbot with multiple AI agents.

Each agent generates useful information.

```text
Research Agent

↓

"Python is popular."
```

```text
Search Agent

↓

"LangGraph is open source."
```

```text
Database Agent

↓

"10 documents found."
```

Without a reducer, only one agent's output survives.

Example:

```python
{
    "messages": [
        "10 documents found."
    ]
}
```

The responses from the other agents are lost.

---

# Another Example: Travel Planner

Three nodes run simultaneously.

```text
Hotel Node

↓

Hotel A
```

```text
Restaurant Node

↓

Restaurant B
```

```text
Attraction Node

↓

Museum C
```

Expected output:

```python
[
    "Hotel A",
    "Restaurant B",
    "Museum C"
]
```

Without a reducer, you might get:

```python
[
    "Museum C"
]
```

Only the last update remains.

---

# Why Doesn't LangGraph Merge Automatically?

LangGraph cannot guess how you want values to be combined.

For example:

Suppose three nodes return numbers.

```python
10
```

```python
20
```

```python
30
```

Should the final value be?

```python
60
```

Or

```python
30
```

Or

```python
[10, 20, 30]
```

Or

```python
Average = 20
```

There is no universally correct answer.

Different applications require different merge strategies.

That's why **you define a reducer**.

---

# The Solution: Reducer

A reducer tells LangGraph exactly **how to merge multiple updates**.

For example:

```python
Current State

["Hello"]

New Update

["How are you?"]

↓

Reducer

↓

["Hello", "How are you?"]
```

Now every update is preserved.

---

# Before vs After Reducer

## Without Reducer

```text
Node A

↓

["Python"]

Node B

↓

["LangGraph"]

Node C

↓

["AI"]

↓

Final State

["AI"]
```

---

## With Reducer

```text
Node A

↓

["Python"]

Node B

↓

["LangGraph"]

Node C

↓

["AI"]

↓

Reducer

↓

["Python", "LangGraph", "AI"]
```

All outputs are safely combined.

---

# Real-World Analogy

Imagine three people editing the same document.

### Without Rules

Person A writes:

```text
Python
```

Person B saves:

```text
LangGraph
```

Person C saves:

```text
AI
```

The final document contains only:

```text
AI
```

Everyone else's work is overwritten.

---

### With a Rule (Reducer)

The rule says:

> "Append new content instead of replacing existing content."

The final document becomes:

```text
Python
LangGraph
AI
```

No information is lost.

---

# Key Takeaways

- LangGraph uses a shared state that can be updated by multiple nodes.
- In parallel workflows, several nodes may update the same state key at the same time.
- Without a reducer, updates can overwrite one another, leading to data loss.
- LangGraph cannot automatically determine the correct way to merge conflicting updates because different applications require different behaviors.
- A reducer provides explicit merge logic, ensuring that all relevant updates are combined correctly.

---

# Summary

The main problem before reducers is **state update conflicts**. When multiple nodes write to the same state field simultaneously, LangGraph has no default way to combine those updates. As a result, some values may overwrite others, causing important information to be lost. Reducers solve this problem by defining how concurrent updates should be merged, making parallel workflows safe, predictable, and reliable.